# 🚗 Smart Parking System
**Automated capacity tracking, spot assignment and fee calculation in Python**

**Team:** Alfred Ernst Falcon · María Montecristo Fontes González · Rubén García Cuesta · Nahún Quintana Granado
**License:** MIT · **Repository:** https://github.com/sabrent-hash/Smart-Parking-System

---
### How to use this notebook
Click **Runtime → Run all**, or press ▶ on each cell from top to bottom.
The code cells use `%%writefile` to save the files exactly as they are in our GitHub repository (`parking_data.py`, `functions.py`, `main.py`). Each team member's function is added to `functions.py` in its own cell (`%%writefile -a` = append).

## 1. Problem and objectives
Manual parking management causes delays, confusion and billing errors. Our modular Python program automates the whole parking lifecycle:

1. **Check** the available slots in real time. *(Alfred)*
2. **Register** a vehicle and assign the **lowest free slot**. *(María)*
3. **Calculate** the exact fee from the parked time. *(Rubén)*
4. **Process** the exit, print the price and free the slot. *(Nahún)*

## 2. Inputs, outputs and worked example

| Inputs | Outputs |
|---|---|
| `parking_slots` (dict `{slot: plate or None}`) | number and list of free slots |
| `license_plate` (str) | assigned slot number (`None` if full) |
| `hours` parked (float, `2.5` = 2 h 30 min) | price in euros + exit message |

**Worked example**
1. Parking with **50 slots, 35 occupied → 15 available**.
2. Car **8492LMP** arrives → assigned **slot #36** (lowest free) → 14 available.
3. It leaves after **2.5 hours** → `2.5 × 60 min × 0.03 €/min =` **4.50 €** → slot #36 freed.

## 3. Initial data — `parking_data.py`
50 slots: slots 1–35 are occupied by demo cars (`0001BCD`, `0002BCD`, …) and slots 36–50 are free (`None`).

In [ ]:
%%writefile parking_data.py
# Copyright (c) 2026 Smart Parking Team (ULPGC)
# Licensed under the MIT License. See LICENSE file for details.
"""Initial data of the Smart Parking System.

The parking has 50 slots. Slots 1 to 35 start occupied by demo cars
(0001BCD, 0002BCD, ...) and slots 36 to 50 start free (None), as in the
worked example of our presentation.
"""

TOTAL_SPOTS = 50       # capacity of the parking
INITIAL_OCCUPIED = 35  # slots occupied at the start

# {slot_number: license_plate or None}
parking_slots: dict[int, str | None] = {
    slot: (f"{slot:04d}BCD" if slot <= INITIAL_OCCUPIED else None)
    for slot in range(1, TOTAL_SPOTS + 1)
}


## 4. `functions.py` — header
Shared import and description of the module. Then each member adds their own function.

In [ ]:
%%writefile functions.py
# Copyright (c) 2026 Smart Parking Team (ULPGC)
# Licensed under the MIT License. See LICENSE file for details.
"""Functions of the Smart Parking System.

Each team member is responsible for one function: Alfred checks the
available spots, María registers the entries, Rubén calculates the
fees and Nahún registers the exits. All of them share the same data
format: parking_slots = {spot_number: license_plate or None}.
"""
import re  # CHANGED: needed to check the license plate format
from collections import defaultdict  # CHANGED: missing import (calculate_fees crashed)

# Spanish license plate format: 4 digits + 3 letters (e.g. 8492LMP)
PLATE_PATTERN = re.compile(r"^[0-9]{4}[A-Z]{3}$")

## 5. Alfred — `available_spots()`
Goes through every slot and returns a **tuple**: the number of free slots and the list of free slot numbers. `(0, [])` means the parking is full.

In [ ]:
%%writefile -a functions.py



# =====================================================================
# ALFRED - available_spots()
# =====================================================================
def available_spots(parking_slots: dict[int, str | None]) -> tuple[int, list[int]]:
    """Checks the number of available parking slots.

    Steps:
        1. Create an empty list and a counter for the free slots.
        2. Go through every slot of the parking.
        3. If a slot is empty (None), add it to the list and count it.
        4. Return the counter and the list of free slots.

    Args:
        parking_slots: All parking slots, with key: slot number and
            value: license plate, or None if the slot is empty.

    Returns:
        A tuple (number of free slots, list of free slot numbers).
        (0, []) means that the parking is full.

    Example:
        >>> available_spots({1: "1234BCD", 2: None, 3: None})
        (2, [2, 3])
    """
    # Step 1: list of free slot numbers and counter of free slots
    available_slots = []
    available_count = 0

    # Steps 2 and 3: loop through the parking slots and count the empty ones
    for slot in parking_slots:
        if parking_slots[slot] is None:
            available_count += 1
            available_slots.append(slot)

    # Step 4: return the count and the list of free slots
    # CHANGED: comment fixed (it returns the count AND the list)
    return available_count, available_slots

In [ ]:
%run functions.py
# Quick test: slots 2 and 3 are free -> expected (2, [2, 3])
print(available_spots({1: "1234BCD", 2: None, 3: None}))

## 6. María — `register_vehicle_entry()`
1. Cleans the plate (`" 8492-lmp "` → `"8492LMP"`).
2. Rejects an empty plate, a plate that is not in the Spanish format **4 digits + 3 letters** (e.g. `8492LMP`), or a car that is already inside (`ValueError`).
3. Uses **Alfred's** `available_spots()` and assigns the **lowest free slot**, or returns `None` if the parking is full.

In [ ]:
%%writefile -a functions.py



# =====================================================================
# MARÍA - register_vehicle_entry()
# =====================================================================
def register_vehicle_entry(parking_slots: dict[int, str | None],
                           license_plate: str) -> int | None:
    """Registers a vehicle entry and assigns it the lowest free slot.

    Steps:
        1. Clean the license plate (remove spaces and hyphens, use uppercase).
        2. Check that the plate is not empty, that it has the Spanish
           format (4 digits + 3 letters, e.g. 8492LMP) and that the car
           is not already inside the parking.
        3. Ask Alfred's available_spots() for the free slots.
        4. If there are no free slots, return None (parking full).
        5. Assign the lowest free slot to the car and return its number.

    Args:
        parking_slots: All parking slots, with key: slot number and
            value: license plate, or None if the slot is empty.
            It is modified in place when the car gets a slot.
        license_plate: License plate of the entering vehicle.

    Returns:
        The number of the assigned parking slot, or None if the
        parking is full.

    Raises:
        ValueError: If the plate is empty, has a wrong format or the car
            is already inside.

    Example:
        >>> slots = {1: "1234BCD", 2: None, 3: None}
        >>> register_vehicle_entry(slots, "8492lmp")
        2
    """
    # Step 1: clean the plate ("8492-lmp" -> "8492LMP")
    plate = license_plate.strip().upper().replace(" ", "").replace("-", "")

    # Step 2: validations  # CHANGED: added (empty plate, format and duplicates)
    if plate == "":
        raise ValueError("License plate cannot be empty.")
    if not PLATE_PATTERN.match(plate):
        raise ValueError(f"Invalid license plate '{plate}'. "
                         "Use 4 digits + 3 letters, e.g. 8492LMP.")
    if plate in parking_slots.values():
        raise ValueError(f"Vehicle {plate} is already inside the parking.")

    # Step 3: use Alfred's function to get the free slots
    # CHANGED: now uses available_spots() so both functions are connected
    free_count, free_slots = available_spots(parking_slots)

    # Step 4: parking full
    if free_count == 0:
        return None

    # Step 5: assign the lowest free slot
    slot_number = min(free_slots)
    parking_slots[slot_number] = plate
    return slot_number

In [ ]:
%run functions.py
# Quick test: slot 1 occupied -> the new car gets slot 2
slots = {1: "1234BCD", 2: None, 3: None}
print(register_vehicle_entry(slots, "8492lmp"))
print(slots)

## 7. Rubén — `calculate_fees()`
Receives a matrix of `[car_id, minutes]` rows and returns `{car_id: total_payment}` rounded to 2 decimals, at **0.03 €/minute**.
If the same car appears several times (several stays), its payments are **added together**. Negative minutes or rows that are not pairs raise a `ValueError`.

In [ ]:
%%writefile -a functions.py



# =====================================================================
# RUBÉN - calculate_fees()
# =====================================================================
default_price_per_minute = 0.03  # euros per minute


def calculate_fees(parking: list[list],
                   price_per_minute: float = default_price_per_minute
                   ) -> dict[str, float]:  # CHANGED: type hints added
    """Calculates the payment for each car in the parking lot.

    Steps:
        1. Go through every row [car_id, minutes].
        2. Skip empty rows and reject rows that are not pairs.
        3. Reject negative minutes.
        4. Add minutes x price to the total of that car (the same car
           can appear several times: several stays).
        5. Return every total rounded to 2 decimals.

    Args:
        parking: matrix (list of pairs) where each row is [car_id, minutes].
            The same car may appear several times (several stays);
            in that case its payments are added together.
        price_per_minute: price charged for each minute of stay.

    Returns:
        dict with {car_id: total_payment}, rounded to 2 decimals.

    Raises:
        ValueError: if any row is not a pair or the minutes are negative.

    Example:
        >>> calculate_fees([["8492LMP", 150]])
        {'8492LMP': 4.5}
    """
    payments = defaultdict(float)

    # Step 1: go through every row  # CHANGED: indentation fixed (4 spaces, was 12)
    for row in parking:
        # Step 2: error cases
        if not row:  # free spot: None or empty list
            continue

        if len(row) != 2:   # Error in case of a list that is not a pair
            raise ValueError(f"Each row must be a pair [id, minutes]: {row}")

        car_id, minutes = row
        if car_id is None:  # free spot: [None, ...]
            minutes = 0

        # Step 3: negative minutes
        if minutes < 0:   # Error in case of a negative number
            raise ValueError(f"Negative minutes for car {car_id}")

        # Step 4: calculation of the payment
        payments[car_id] += minutes * price_per_minute

    # Step 5: dictionary assembly rounded to two decimals  # CHANGED: typo "Diccionary"
    return {car_id: round(total, 2) for car_id, total in payments.items()}

In [ ]:
%run functions.py
# Car 101 appears twice: 45 + 30 = 75 min -> 2.25 €
print(calculate_fees([[101, 45], [102, 120], [101, 30]]))

## 8. Nahún — `register_vehicle_exit()`
Checks that the car is inside and that the hours are not negative, converts the hours to minutes, gets the price with **Rubén's** `calculate_fees()`, **frees the slot** and prints the price.

In [ ]:
%%writefile -a functions.py



# =====================================================================
# NAHÚN - register_vehicle_exit()
# =====================================================================
def register_vehicle_exit(parking_slots: dict[int, str | None],
                          license_plate: str,
                          hours: float) -> dict[int, str | None]:
    """Frees the car's spot, prints the price and returns the parking.

    Steps:
        1. Clean the license plate (remove spaces and hyphens, use uppercase).
        2. Check that the car is in the parking and that the hours
           are not negative.
        3. Convert the hours to minutes and get the price with
           Rubén's calculate_fees().
        4. Find the car's slot, free it and print the receipt.

    Args:
        parking_slots: All parking slots, with key: slot number and
            value: license plate, or None if the slot is empty.
        license_plate: License plate of the vehicle that is leaving.
        hours: Time parked, in decimal hours (2.5 = 2 h 30 min).

    Returns:
        The updated parking_slots (the car's slot is set to None).
        If there is an error, the parking is returned unchanged.

    Example:
        >>> register_vehicle_exit({36: "8492LMP"}, "8492LMP", 2.5)
        Vehicle 8492LMP left spot 36. Total price: 4.50 €
        {36: None}
    """
    # Step 1: clean the plate  # CHANGED: same format as the entry
    license_plate = license_plate.strip().upper().replace(" ", "").replace("-", "")

    # Step 2: error cases
    # Error: plate not found
    if license_plate not in parking_slots.values():
        print(f"Error: license plate {license_plate} not found in the parking.")
        return parking_slots

    # Error: negative hours
    if hours < 0:
        print("Error: hours cannot be negative.")
        return parking_slots

    # Step 3: calculate the price
    # CHANGED: parking_fee() did not exist. Rubén's calculate_fees() uses minutes.
    total_price = calculate_fees([[license_plate, hours * 60]])[license_plate]

    # Step 4: free the spot
    for spot, plate in parking_slots.items():
        if plate == license_plate:
            parking_slots[spot] = None
            # CHANGED: :.2f so the price always has 2 decimals (4.50 €)
            print(f"Vehicle {license_plate} left spot {spot}. Total price: {total_price:.2f} €")
            return parking_slots


In [ ]:
%run functions.py
# Worked example: 8492LMP parked 2.5 h in slot 36 -> 4.50 €
print(register_vehicle_exit({36: "8492LMP"}, "8492LMP", 2.5))

## 9. Integration — `main.py`
The user types a license plate:
- If the car is **not inside** → **ENTRY** with María's function (which uses Alfred's).
- If the car **is inside** → **EXIT**: asks the hours and calls Nahún's function, which prints the **price** using Rubén's.
- After every operation, the free slots are shown with Alfred's function.
- When the user quits (`q`), a **daily report** shows what each car paid and the total income (Rubén's function adds several stays of the same car).

Wrong data (invalid plate, hours that are not a number, negative hours) shows an error message without stopping the program.

In [ ]:
%%writefile main.py
# Copyright (c) 2026 Smart Parking Team (ULPGC)
# Licensed under the MIT License. See LICENSE file for details.
"""Smart Parking System - main program.

Interactive program that connects the four team functions: the user
types a license plate; if the car is not inside it is an ENTRY
(register_vehicle_entry), and if it is inside it is an EXIT
(register_vehicle_exit, which uses calculate_fees to print the price).
After every operation the free slots are shown with available_spots,
and when the user quits, the income of the day is shown.
Run it with: python main.py  (type 'q' to quit).
"""
from parking_data import parking_slots  # CHANGED: "datasets" -> "parking_data"
from functions import (PLATE_PATTERN, available_spots, calculate_fees,
                       register_vehicle_entry, register_vehicle_exit)
# CHANGED: removed "user = True" (it was never used)


def clean_plate(text: str) -> str:
    """Cleans a license plate typed by the user.

    Uses the same format as the team functions: no spaces, no hyphens
    and uppercase letters ("8492-lmp" -> "8492LMP").

    Args:
        text: Plate as typed by the user.

    Returns:
        The clean plate.
    """
    return text.strip().upper().replace(" ", "").replace("-", "")


def show_status(parking_slots: dict[int, str | None]) -> None:
    """Prints how many slots are free, using Alfred's available_spots().

    Args:
        parking_slots: All parking slots {slot number: plate or None}.
    """
    free_count, _ = available_spots(parking_slots)
    print(f"Total: {len(parking_slots)} | Occupied: {len(parking_slots) - free_count}"
          f" | Available: {free_count}")


def ask_hours() -> float | None:
    """Asks how many hours the car was parked and checks the answer.

    Steps:
        1. Read the text typed by the user.
        2. Accept "2.5" and also "2,5" (Spanish decimal comma).
        3. If it is not a number, print an error and return None.

    Returns:
        The hours as a float, or None if the input is not a number.
    """
    text = input("Hours parked (e.g. 2.5): ").strip().replace(",", ".")
    try:
        return float(text)
    except ValueError:  # CHANGED: before, a wrong number crashed the program
        print("Error: hours must be a number, e.g. 2.5")
        return None


def print_daily_report(stays_log: list[list]) -> None:
    """Prints how much each car paid and the total income of the day.

    Rubén's calculate_fees() adds together all the stays of the same
    car, so a car that parked twice appears once with its total.

    Args:
        stays_log: Finished stays as [plate, minutes] rows.
    """
    print("\n--- Daily report ---")
    if not stays_log:
        print("No car has left the parking yet. Income: 0.00 €")
        return
    totals = calculate_fees(stays_log)
    for plate, total in totals.items():
        print(f"  {plate}: {total:.2f} €")
    print(f"  TOTAL INCOME: {sum(totals.values()):.2f} €")


def main() -> None:
    """Runs the parking until the user types 'q'.

    Steps:
        1. Show the instructions and the initial state of the parking.
        2. Ask for a license plate ('q' to quit) and check its format
           (4 digits + 3 letters). Wrong plates are rejected here, both
           for entries and exits.
        3. If the car is inside -> EXIT: ask the hours and call Nahún's
           register_vehicle_exit(), which prints the price using Rubén's
           calculate_fees(). The stay is saved for the daily report.
        4. If the car is not inside -> ENTRY: call María's
           register_vehicle_entry() (which uses Alfred's available_spots()).
        5. Show the new state and repeat.
        6. When the user quits, print the daily report.
    """
    stays_log: list[list] = []  # finished stays [plate, minutes]

    # Step 1: instructions and initial state
    print("=== SMART PARKING SYSTEM ===")
    print("Type a NEW plate to ENTER, or a plate that is INSIDE to EXIT and pay.")
    print("Plate format: 4 digits + 3 letters (e.g. 8492LMP).")
    show_status(parking_slots)

    while True:
        # Step 2: read the plate
        plate = clean_plate(input("\nLicense plate (or 'q' to quit): "))

        if plate == "Q":
            # Step 6: daily report
            print_daily_report(stays_log)
            print("Goodbye!")
            break

        if plate == "":
            print("Error: License plate cannot be empty.")
            continue

        # CHANGED: the format is checked here too, for ENTRIES and EXITS
        if not PLATE_PATTERN.match(plate):
            print(f"Error: Invalid license plate '{plate}'. "
                  "Use 4 digits + 3 letters, e.g. 8492LMP.")
            continue

        if plate in parking_slots.values():
            # Step 3: the car is inside, so this is an EXIT
            print(f"Vehicle {plate} is inside -> EXIT")
            hours = ask_hours()
            if hours is None:
                continue
            register_vehicle_exit(parking_slots, plate, hours)
            if plate not in parking_slots.values():  # the exit was correct
                stays_log.append([plate, hours * 60])
        else:
            # Step 4: the car is not inside, so this is an ENTRY
            try:
                spot = register_vehicle_entry(parking_slots, plate)
            except ValueError as error:
                print(f"Error: {error}")
                continue
            if spot is None:
                print("Sorry, the parking is full.")
            else:
                print(f"Vehicle {plate} assigned to spot {spot}. "
                      f"(Type {plate} again when it leaves.)")

        # Step 5: new state of the parking
        show_status(parking_slots)


if __name__ == "__main__":
    main()


## 10. 🎬 Live demo — interactive program
Run the cell and type license plates in the box that appears:
- A **new** plate (e.g. `8492LMP`) → it **enters** and gets a slot.
- The **same** plate again → it **exits**: type the hours (e.g. `2.5`) and the **price** appears.
- Try wrong data: an invalid plate (`ABC123`), `abc` as hours, or `-1` hours.
- Type **`q`** to quit and see the **daily report** with the total income.

Each time you run this cell the parking starts again from 35/50.

In [ ]:
import importlib, parking_data
importlib.reload(parking_data)   # start again from the initial parking (35/50)
%run main.py

## 11. ✍️ Try it yourself
Change the license plate and the hours, then run the cell. The car enters (gets the lowest free slot) and then leaves after the hours you chose.

Try wrong data too: `plate = "ABC123"` (wrong format), `plate = "0001BCD"` (a car that is already inside) or `hours = -1` (negative hours).

In [ ]:
import importlib, parking_data, functions
importlib.reload(parking_data)   # start again from the initial parking (35/50)
importlib.reload(functions)
from parking_data import parking_slots
from functions import available_spots, register_vehicle_entry, register_vehicle_exit

# ---- Change these values ----
plate = "8492LMP"
hours = 2.5          # 2.5 = 2 h 30 min
# -----------------------------

def status():
    free, _ = available_spots(parking_slots)
    print(f"Total: {len(parking_slots)} | Occupied: {len(parking_slots) - free} | Available: {free}")

status()

# ENTRY (María's function, which uses Alfred's)
try:
    spot = register_vehicle_entry(parking_slots, plate)
except ValueError as error:
    print(f"Error: {error}")
else:
    if spot is None:
        print("Sorry, the parking is full.")
    else:
        print(f"\nENTRY: vehicle {plate.strip().upper()} assigned to spot {spot}.")
        status()

        # EXIT (Nahún's function, which uses Rubén's)
        print(f"\nEXIT after {hours} h:")
        register_vehicle_exit(parking_slots, plate, hours)
        status()

## 12. Results and conclusions
- Every car gets the **lowest free slot**, and freed slots are reused immediately.
- The price matches the worked example: **2.5 h → 4.50 €**.
- The four functions are connected: María's uses Alfred's, and Nahún's uses Rubén's.
- Invalid plates and wrong hours are detected and reported without stopping the program.
- The daily report adds several stays of the same car (e.g. **8492LMP: 4.50 + 1.80 = 6.30 €**).

**What we learned:** by combining modular Python design with a collaborative Git workflow and Google Colab, we turned individual functions into shared, open-source software.

**License:** MIT — see the `LICENSE` file in the repository.